In [138]:
from io import BytesIO
import os
from pathlib import Path
import sqlite3
import tarfile

import pandas as pd
from termcolor import colored
from tqdm import tqdm

In [127]:
FTS3_SRV = 'https://fts3-public.cern.ch:8446'
#FTS3_SRV = 'https://fts3-public.fnal.gov:8446'
NERSC_BASE_URL = 'davs://dtn14.nersc.gov:1094'
FNAL_DEST = 'davs://fndcadoor.fnal.gov:2880/dune/scratch/dunepro/ingest/neardet-2x2/pre_dropbox'
CRS_BASE_DIR = '/global/cfs/cdirs/dune/www/data/2x2/CRS.run2/ColdOperations/data/2025_Operations_Cold'
LRS_BASE_DIR = '/global/cfs/cdirs/dune/www/data/2x2/LRS_run2'

In [167]:
def mini(df):
    return df[['run', 'subrun', 'filename', 'nersc_path']]

def read_metacat_list(listfile):
    with open(listfile) as f:
        return set(l.strip().split(':')[1] for l in f)

def sanity_check(srcpath):
    if not srcpath.exists():
        print(colored(f'MISSING {srcpath}', 'red'))
        return False
    if not Path(f'{srcpath}.json').exists():
        print(colored(f'META? {srcpath}', 'blue'))
        return False
    if srcpath.stat().st_size > 10e9:
        print(f'BIG {srcpath}')
        return False
    return True

def _maybe_load_list_file(path_or_collection):
    if type(path_or_collection) not in [str, Path]:
        return path_or_collection
    with open(path_or_collection) as f:
        return [l.strip() for l in f.readlines()]

def dump_transfers(df, outpath, nersc_paths, nersc_base_dir,
                   selection=[]):
    df = df[df['nersc_path'].isin(set(nersc_paths))]
    selection = set(fname.removesuffix('.json')
                    for fname in _maybe_load_list_file(selection))
    lines = []
    for fname, nersc_path in zip(df['filename'], df['nersc_path']):
        if selection and (fname not in selection):
            continue
        srcpath = Path(nersc_base_dir) / nersc_path / fname
        if not sanity_check(srcpath):
            continue
        cmd = f'fts-rest-transfer-submit -s {FTS3_SRV} {NERSC_BASE_URL}{srcpath} {FNAL_DEST}/{srcpath.name}'
        lines.append(cmd)
    with open(outpath, 'w') as outfile:
        outfile.write('\n'.join(lines) + '\n')

def dump_txQ(outpath, nersc_paths, **kw):
    global dfq2do
    dump_transfers(dfq2do, outpath, nersc_paths, CRS_BASE_DIR, **kw)

def dump_txL(outpath, nersc_paths, **kw):
    global dfl2do
    dump_transfers(dfl2do, outpath, nersc_paths, LRS_BASE_DIR, **kw)

def tar_metadata(df, tarpath, nersc_paths, nersc_base_dir):
    buffer = BytesIO()
    with tarfile.open(fileobj=buffer, mode='w:gz') as tar:
        df = df[df['nersc_path'].isin(set(nersc_paths))]
        for fname, nersc_path in tqdm(list(zip(df['filename'], df['nersc_path']))):
            srcpath = Path(nersc_base_dir) / nersc_path / fname
            if not sanity_check(srcpath):
                continue
            jsonpath = Path(f'{srcpath}.json')
            tar.add(jsonpath, arcname=jsonpath.name)
    with open(tarpath, 'wb') as f:
        f.write(buffer.getvalue())

def tar_mdQ(tarpath, nersc_paths):
    global dfq2do
    tar_metadata(dfq2do, tarpath, nersc_paths, CRS_BASE_DIR)

def tar_mdL(tarpath, nersc_paths):
    global dfl2do
    tar_metadata(dfl2do, tarpath, nersc_paths, LRS_BASE_DIR)

In [72]:
conn = sqlite3.connect('/global/cfs/cdirs/dune/www/data/2x2/DB/RunsDB/latest/run2/2x2runs_run2.latest.sqlite')

In [186]:
dfq = pd.read_sql_query('select * from CRS_summary', conn)
#dfq['filename'] = dfq['filename'].map(lambda s: s.replace('.h5', '.hdf5'))
dfq['nersc_path'] = dfq['nersc_path'].map(lambda s: s.replace('/dvs_ro/cfs/cdirs/dune/www/data/2x2/nearline_run2/packet/ColdOperations/data/2025_Operations_Cold/', ''))
dfq['nersc_path'] = dfq['nersc_path'].map(lambda s: os.path.dirname(s))
dfq = mini(dfq)

dfl = pd.read_sql_query('select * from LRS_summary', conn)
#dfl['filename'] = dfl['filename'].map(lambda s: s.replace('mpd_run_data', 'mpd_run_run2data'))
dfl['nersc_path'] = dfl['nersc_path'].map(lambda s: s.replace('/dvs_ro/cfs/cdirs/dune/www/data/2x2/LRS_run2/', ''))
dfl['nersc_path'] = dfl['nersc_path'].map(lambda s: os.path.dirname(s))
dfl = mini(dfl)

# metacat query "files where core.run_type=neardet-2x2-lar-charge and core.runs[any]>=60000"
qok = read_metacat_list('../data-mgmt/2x2_run2_inventory/all_charge.txt')
qok = {f.replace('.hdf5', '.h5') for f in qok}

# metacat query "files where core.run_type=neardet-2x2-lar-light and core.runs[any]>=60000"
lok = read_metacat_list('../data-mgmt/2x2_run2_inventory/all_light.txt')
lok = {f.replace('mpd_run_run2data', 'mpd_run_data') for f in lok}

dfq2do = dfq[~dfq['filename'].isin(qok)]
dfq4sho = dfq[dfq['filename'].isin(qok)]

dfl2do = dfl[~dfl['filename'].isin(lok)]
dfl4sho = dfl[dfl['filename'].isin(lok)]

In [22]:
dfq['filename'].value_counts()

filename
binary-0060011-2025_10_15_05_26_48_CDT.h5    2
binary-0060011-2025_10_15_05_24_17_CDT.h5    2
binary-0060011-2025_10_15_05_21_46_CDT.h5    2
binary-0060011-2025_10_15_05_19_15_CDT.h5    2
binary-0060011-2025_10_15_05_16_44_CDT.h5    2
                                            ..
binary-0060022-2025_10_18_19_24_21_CDT.h5    1
binary-0060022-2025_10_18_19_26_52_CDT.h5    1
binary-0060022-2025_10_18_19_29_23_CDT.h5    1
binary-0060022-2025_10_18_19_31_54_CDT.h5    1
binary-0060022-2025_10_18_19_11_46_CDT.h5    1
Name: count, Length: 9362, dtype: int64

In [92]:
dfl2do['nersc_path'].value_counts()

nersc_path
source_rn_bin1/injection                                                                                 2673
source_ambe_bin0/mod2_pmt_trig                                                                            896
cosmics_bin0                                                                                              844
source_na22_bin1                                                                                          809
cosmics_bin7                                                                                              728
source_ambe_bin3/two_trig_9600ns_window                                                                   723
cosmics_bin5                                                                                              688
cosmics_bin14                                                                                             569
source_co60_colimated_bin1                                                                                551

In [150]:
txQ = '../data-mgmt/2x2_run2_inventory/transfers/transfers_charge.sh'
txL = '../data-mgmt/2x2_run2_inventory/transfers/transfers_light.sh'
tarQ = '../data-mgmt/2x2_run2_inventory/metadata_charge.tar.gz'
tarL = '../data-mgmt/2x2_run2_inventory/metadata_light.tar.gz'

# leftover json files in the dropbox
sel = '../data-mgmt/2x2_run2_inventory/failed_transfers.txt'

In [147]:
s = '''
source_ambe_bin0/mod2_pmt_trig_nosource                                                                   186
source_ambe_bin3/two_trig_243us_period                                                                    174
source_ambe_bin3/two_trig_147us_period                                                                    168
source_ambe_bin3/two_trig_291us_period                                                                    166
source_ambe_bin3/two_trig_51us_period                                                                     166
source_na22_bin0/threshold_trig                                                                           164
source_ambe_bin3/two_trig_32us_period_500tick                                                             163
cosmics_bin6                                                                                              162
source_ambe_bin3/two_trig_195us_period                                                                    161
source_y88_bin2/cosmics_same_thd                                                                          159
source_co60_uncolimated_bin2/cosmics_same_thd                                                             136
cold_commission/48V_VoltScan                                                                              133
'''

In [148]:
nersc_paths = [l.split()[0] for l in s.strip().split('\n')]

In [149]:
nersc_paths

['source_ambe_bin0/mod2_pmt_trig_nosource',
 'source_ambe_bin3/two_trig_243us_period',
 'source_ambe_bin3/two_trig_147us_period',
 'source_ambe_bin3/two_trig_291us_period',
 'source_ambe_bin3/two_trig_51us_period',
 'source_na22_bin0/threshold_trig',
 'source_ambe_bin3/two_trig_32us_period_500tick',
 'cosmics_bin6',
 'source_ambe_bin3/two_trig_195us_period',
 'source_y88_bin2/cosmics_same_thd',
 'source_co60_uncolimated_bin2/cosmics_same_thd',
 'cold_commission/48V_VoltScan']

In [139]:
tar_mdL(tarL, nersc_paths)

100%|██████████| 1802/1802 [00:19<00:00, 92.75it/s] 


In [140]:
!ls -lh $tarL

-rw-r--r-- 1 mkramer mkramer 124K Jan 31 16:26 ../data-mgmt/2x2_run2_inventory/metadata_light.tar.gz


In [142]:
!tar tf $tarL | wc -l

1802


In [143]:
counts = [int(l.split()[1]) for l in s.strip().split('\n')]
sum(counts)

1802

In [168]:
dump_txL(txL, nersc_paths, selection=selL)